# The Phase-Locked Loop: The Circuit That Built Modern Communications

The phase-locked loop (PLL) is one of the most important building blocks of modern communications and electronics. In a modern smartphone, there are probably in the hundreds of PLLs, some implemented as analog circuits and some through digital signal processing (DSP). In this note, we will explain why PLLs are needed, then what they do, and finally how to implement them in DSP.

## Why are PLLs needed?

The beginning of wireless communications in 1896 used what we would today call on-off keying (OOK), a special form of amplitude modulation (AM) in which only two amplitudes are used--on (full carrier power) and off (no carrier). [You should recognize this from the ADS-B lab.](https://rfsoc.mit.edu/6S965/F25/assignments/week08) By 1906, technology had progressed to analog AM, allowing for continuously-variable amplitude. Neither of these things required a PLL. Why? Let's examine what an OOK/AM signal looks like:

In [32]:
import numpy as np

sample_frequency_hz = 1e3 # 1 kSps sampling frequency

# Start by defining our message, in this case a 4-symbol-long message ON-OFF-ON-OFF or 0b1010
message = np.array([1, 0, 1, 0], dtype = np.int16)

# Create our baseband signal from the message by making each symbol last a certain amount of time.
symbol_length_samples = 100 # Each symbol lasts 100 samples
baseband = np.repeat(message, symbol_length_samples) # Repeat each symbol symbol_length_samples times

num_samples = len(baseband)

def calculate_am_waveforms(params):
    # Create our carrier signal and modulate it
    carrier_frequency_discrete = params["carrier_frequency_hz"] / sample_frequency_hz
    carrier = np.e ** (1j * 2 * np.pi * carrier_frequency_discrete * np.arange(num_samples))
    modulated_carrier = baseband * carrier

    return {"Baseband": baseband, "Re(Modulated Carrier)": modulated_carrier.real, "Im(Modulated Carrier)": modulated_carrier.imag}

#=============================================IGNORE EVERYTHING BELOW HERE===========================================

%matplotlib ipympl
import matplotlib.pyplot as plt
from ipywidgets import AppLayout, FloatSlider, IntSlider, Layout, Box

plt.ioff() # Make Matplotlib only update when explicitly asked to

# Build the user interface
fig, (left_axes, right_axes) = plt.subplots(1, 2)
fig.canvas.header_visible = False
fig.canvas.layout.min_height = '400px'
fig.canvas.layout.min_width = '800px'

carrier_frequency_slider = FloatSlider(
    orientation='horizontal',
    description='Carrier Frequency (Hz):',
    value=10,
    min=10,
    max=100
)

timestep_slider = IntSlider(
    orientation='horizontal',
    description='Timestep:',
    value = 0,
    min = 0,
    max = num_samples - 1
)

vbox_layout = Layout(display = "flex",
                     flex_flow = "column",
                     align_items = "stretch",
                     width = "100%"
                    )
vbox = Box(children = [carrier_frequency_slider, timestep_slider], layout = vbox_layout)

# Function that gets called whenever sliders move
lines = {}
timestep_cursor = None
iq_vector = None
first = True

def update_visualization(change):
    global first
    global timestep_cursor
    global iq_vector
    params = {"carrier_frequency_hz": carrier_frequency_slider.value}
    waveforms = calculate_am_waveforms(params)
    
    if first:
        for waveform_name, waveform in waveforms.items():
            lines[waveform_name] = left_axes.plot(waveform, label = waveform_name)
        # Plot time-domain signals on left axes
        left_axes.legend()
        left_axes.set_xlabel("Timestep (samples)")
        left_axes.set_ylabel("Amplitude (volts)")
        # Plot timestep cursor on left axes (vertical line that shows what time we are viewing)
        timestep_cursor = left_axes.axvline(timestep_slider.value, color = 'r')
        # Plot I/Q in complex plane on right axes
        iq_vector = right_axes.plot([0, 0], [0, 0], "-o")
        right_axes.set_xlabel("Re")
        right_axes.set_ylabel("Im")
        right_axes.set_xlim(-1, 1)
        right_axes.set_ylim(-1, 1)
        right_axes.spines['left'].set_position('center')
        right_axes.spines['bottom'].set_position('center')
        right_axes.spines['right'].set_color('none')
        right_axes.spines['top'].set_color('none')
        right_axes.xaxis.set_ticks_position('bottom')
        right_axes.yaxis.set_ticks_position('left')
        first = False
    else:
        timestep_cursor.set_xdata([timestep_slider.value])
        for waveform_name, waveform in waveforms.items():
            lines[waveform_name][0].set_data(np.arange(len(waveform)), waveform)
        am_waveform_real = waveforms["Re(Modulated Carrier)"]
        am_waveform_imag = waveforms["Im(Modulated Carrier)"]
        iq_vector[0].set_data([0, am_waveform_real[timestep_slider.value]], [0, am_waveform_imag[timestep_slider.value]])

    fig.canvas.draw()
    fig.canvas.flush_events()

# Set up visualization
plt.title("Amplitude Modulation Interactive Demo")

# Update visualization whenever sliders move
carrier_frequency_slider.observe(update_visualization, names = "value")
timestep_slider.observe(update_visualization, names = "value")

# Create layout with plot above sliders
AppLayout(
    center = fig.canvas,
    footer = vbox,
    pane_heights = [0, 6, 1]
)

AppLayout(children=(Box(children=(FloatSlider(value=10.0, description='Carrier Frequency (Hz):', min=10.0), In…